# LeetCode 1390 — Four Divisors bằng Python

Link đề: https://leetcode.com/problems/four-divisors/ · Độ khó: **Medium** · Chủ đề: Array, Math, Prime Factorization, Sieve Theory

Skeleton LeetCode:

```python
class Solution:
    def sumFourDivisors(self, nums: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Liệt kê ước của một số hiệu quả bằng cách duyệt **tới căn bậc hai** (ước đi theo cặp `d` và `x // d`), xử lý đúng trường hợp **số chính phương**.
2. Biết **dừng sớm** khi đã thấy quá 4 ước — tối ưu đơn giản nhưng hiệu quả.
3. Dùng lý thuyết số: số có **đúng 4 ước** ⇔ `x = p³` hoặc `x = p·q` (`p ≠ q` nguyên tố).
4. Cài **sàng Eratosthenes / sàng ước nguyên tố nhỏ nhất (SPF)** để phân tích thừa số nhanh.
5. Biết kỹ thuật **sàng điều hòa** (harmonic sieve) tính số ước và tổng ước cho mọi số tới `M` trong `O(M log M)`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **tổng các ước** của những số trong mảng có **đúng bốn ước**. Nếu không có số nào như vậy, trả về 0. (Số xuất hiện nhiều lần thì cộng nhiều lần.)

- **Ví dụ 1:** `nums = [21,4,7]` → `32`. 21 có 4 ước `1, 3, 7, 21` (tổng 32); 4 có 3 ước; 7 có 2 ước.
- **Ví dụ 2:** `nums = [21,21]` → `64`.
- **Ví dụ 3:** `nums = [1,2,3,4,5]` → `0`.

**Ràng buộc:** `1 <= nums.length <= 10^4`; `1 <= nums[i] <= 10^5`.

In [1]:
import math
import random
import time

EXAMPLES = [([21, 4, 7], 32), ([21, 21], 64), ([1, 2, 3, 4, 5], 0)]
MAXV = 10**5
for nums, want in EXAMPLES:
    print(f"nums = {nums} -> mong đợi {want}")

nums = [21, 4, 7] -> mong đợi 32
nums = [21, 21] -> mong đợi 64
nums = [1, 2, 3, 4, 5] -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Ước số và cặp ước

`d` là ước của `x` nếu `x % d == 0`. Nếu `d` là ước thì `x // d` cũng là ước, và trong mỗi cặp có **ít nhất một** số `≤ √x`. Vì vậy chỉ cần duyệt `d` từ 1 tới `⌊√x⌋`. Nếu `d * d == x` (số chính phương) thì cặp chỉ có **một** số — phải đếm một lần.

In [2]:
def divisors_pairs(x):
    out = []
    d = 1
    while d * d <= x:
        if x % d == 0:
            out.append((d, x // d))
        d += 1
    return out

print("21:", divisors_pairs(21))
print("36:", divisors_pairs(36), "<- (6, 6) chỉ là MỘT ước")

21: [(1, 21), (3, 7)]
36: [(1, 36), (2, 18), (3, 12), (4, 9), (6, 6)] <- (6, 6) chỉ là MỘT ước


### 3.2 `math.isqrt` — căn bậc hai nguyên chính xác

`math.isqrt(x)` trả `⌊√x⌋` dạng số nguyên, không sai số dấu phẩy động như `int(x ** 0.5)` (với số rất lớn).

In [3]:
for x in [24, 25, 99999, 10**5]:
    print(x, math.isqrt(x), int(x ** 0.5))

24 4 4
25 5 5
99999 316 316
100000 316 316


### 3.3 Số ước theo phân tích thừa số nguyên tố

Nếu `x = p1^a1 · p2^a2 · … · pk^ak` thì số ước `τ(x) = (a1+1)(a2+1)…(ak+1)`. `τ(x) = 4` chỉ xảy ra khi:
- một thừa số với số mũ 3: `x = p³` (`3 + 1 = 4`), ước là `1, p, p², p³`;
- hai thừa số với số mũ 1: `x = p·q`, `p ≠ q` (`2·2 = 4`), ước là `1, p, q, pq`.

In [4]:
def factorize(x):
    f, p = {}, 2
    while p * p <= x:
        while x % p == 0:
            f[p] = f.get(p, 0) + 1
            x //= p
        p += 1
    if x > 1:
        f[x] = f.get(x, 0) + 1
    return f

for x in [21, 8, 12, 49, 30]:
    f = factorize(x)
    print(x, f, "-> τ =", math.prod(a + 1 for a in f.values()))

21 {3: 1, 7: 1} -> τ = 4
8 {2: 3} -> τ = 4
12 {2: 2, 3: 1} -> τ = 6
49 {7: 2} -> τ = 3
30 {2: 1, 3: 1, 5: 1} -> τ = 8


### 3.4 Sàng Eratosthenes và sàng ước nguyên tố nhỏ nhất (SPF)

Sàng SPF: `spf[x]` = ước nguyên tố nhỏ nhất của `x`. Với mỗi số nguyên tố `p`, đánh dấu `spf[m] = p` cho các bội `m` (bắt đầu từ `p²`) chưa có giá trị. Sau đó `x` là nguyên tố ⇔ `spf[x] == x`. Chi phí `O(M log log M)`.

In [5]:
def build_spf(M):
    spf = list(range(M + 1))           # ban đầu spf[x] = x
    for p in range(2, math.isqrt(M) + 1):
        if spf[p] == p:                 # p nguyên tố
            for m in range(p * p, M + 1, p):
                if spf[m] == m:
                    spf[m] = p
    return spf

spf_small = build_spf(50)
print("spf 2..20:", spf_small[2:21])
print("nguyên tố ≤ 50:", [x for x in range(2, 51) if spf_small[x] == x])

spf 2..20: [2, 3, 2, 5, 2, 7, 2, 3, 2, 11, 2, 13, 2, 3, 2, 17, 2, 19, 2]
nguyên tố ≤ 50: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


### 3.5 Sàng điều hòa: cộng `d` vào mọi bội của `d`

Duyệt `d = 1..M`, với mỗi bội `m = d, 2d, 3d, …` thì `cnt[m] += 1`, `sm[m] += d`. Tổng số thao tác `M/1 + M/2 + … + M/M ≈ M ln M` (chuỗi điều hòa).

In [6]:
M = 30
cnt, sm = [0] * (M + 1), [0] * (M + 1)
for d in range(1, M + 1):
    for m in range(d, M + 1, d):
        cnt[m] += 1
        sm[m] += d
print("số có đúng 4 ước ≤ 30:", [(x, sm[x]) for x in range(1, M + 1) if cnt[x] == 4])
print("số thao tác ≈", sum(M // d for d in range(1, M + 1)), "| M ln M ≈", round(M * math.log(M)))

số có đúng 4 ước ≤ 30: [(6, 12), (8, 15), (10, 18), (14, 24), (15, 24), (21, 32), (22, 36), (26, 42), (27, 40)]
số thao tác ≈ 111 | M ln M ≈ 102


## 4. Ý tưởng nền tảng / trực giác

Tính tay vài số:

| x | ước | số ước | dạng | đóng góp |
|---|---|---|---|---|
| 21 | 1, 3, 7, 21 | 4 | 3·7 (p·q) | 32 |
| 4 | 1, 2, 4 | 3 | 2² | 0 |
| 7 | 1, 7 | 2 | nguyên tố | 0 |
| 8 | 1, 2, 4, 8 | 4 | 2³ (p³) | 15 |
| 12 | 1, 2, 3, 4, 6, 12 | 6 | 2²·3 | 0 |
| 49 | 1, 7, 49 | 3 | 7² | 0 |
| 1 | 1 | 1 | – | 0 |

Hai hướng tiếp cận:
1. **Đếm ước trực tiếp** (duyệt tới `√x`, dừng sớm khi > 4).
2. **Nhận dạng dạng số** `p³` hoặc `p·q` bằng phân tích thừa số (nhanh nhờ sàng).

## 5. Cách 1 — Brute Force / baseline: thử mọi `d` từ 1 tới `x`

In [7]:
def sum_four_brute(nums):
    total = 0
    for x in nums:
        divs = [d for d in range(1, x + 1) if x % d == 0]
        if len(divs) == 4:
            total += sum(divs)
    return total

for nums, want in EXAMPLES:
    print(sum_four_brute(nums), "(mong đợi", want, ")")

32 (mong đợi 32 )
64 (mong đợi 64 )
0 (mong đợi 0 )


**Độ phức tạp:** mỗi số `x` tốn `O(x)` ⇒ worst `n · M = 10^4 · 10^5 = 10^9` phép — quá chậm trong Python.

**Nút thắt:** ta thử cả những `d > √x`, trong khi mỗi ước lớn đã "lộ diện" qua ước nhỏ tương ứng `x // d`. Đo thử với 200 số lớn:

In [8]:
rng = random.Random(0)
sample = [rng.randint(90_000, 100_000) for _ in range(200)]
t0 = time.perf_counter()
sum_four_brute(sample)
dt = time.perf_counter() - t0
print(f"200 số ~10^5: {dt:.3f}s -> 10^4 số: ~{dt * 50:.0f}s")

200 số ~10^5: 0.718s -> 10^4 số: ~36s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cặp ước):** duyệt `d ≤ √x`; mỗi `d` chia hết cho ta hai ước `d` và `x // d` (một ước nếu bằng nhau) ⇒ `O(√x)` mỗi số ≈ 316 bước với `x = 10^5`.

**Insight 2 (dừng sớm):** chỉ quan tâm "có đúng 4 ước không" ⇒ ngay khi đếm được **hơn 4** thì bỏ số đó.

**Insight 3 (lý thuyết số):** `τ(x) = 4 ⇔ x = p³ hoặc x = p·q (p ≠ q nguyên tố)`. Với sàng SPF:
- `p = spf[x]`, `q = x // p`.
- Nếu `p < q` và `q` nguyên tố (`spf[q] == q`) ⇒ `x = p·q`, tổng ước `1 + p + q + x`. (Điều kiện `p < q` vừa loại `q = p`, vừa loại `q = 1` khi `x` là số nguyên tố — vì `spf[1] == 1` cũng "trông như" nguyên tố!)
- Nếu `x == p³` ⇒ tổng ước `1 + p + p² + p³`.
- Ngược lại ⇒ không phải 4 ước.

**Quy tắc "nếu… thì…":** *nếu `d·d == x` thì chỉ cộng một ước* — quên điều này là lỗi số 1 của bài.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt ước tới `√x` cho từng số, dừng sớm

In [9]:
def four_div_sum(x):
    """Trả tổng ước nếu x có đúng 4 ước, ngược lại 0."""
    cnt = s = 0
    d = 1
    while d * d <= x:
        if x % d == 0:
            e = x // d
            if d == e:
                cnt += 1; s += d
            else:
                cnt += 2; s += d + e
            if cnt > 4:                 # dừng sớm
                return 0
        d += 1
    return s if cnt == 4 else 0

def sum_four_sqrt(nums):
    return sum(four_div_sum(x) for x in nums)

for nums, want in EXAMPLES:
    print(sum_four_sqrt(nums), "(mong đợi", want, ")")

32 (mong đợi 32 )
64 (mong đợi 64 )
0 (mong đợi 0 )


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def sumFourDivisors(self, nums: list[int]) -> int:
        total = 0
        for x in nums:
            cnt = s = 0
            d = 1
            while d * d <= x:
                if x % d == 0:
                    e = x // d
                    if d == e:
                        cnt += 1; s += d
                    else:
                        cnt += 2; s += d + e
                    if cnt > 4:
                        break
                d += 1
            if cnt == 4:
                total += s
        return total

print([Solution().sumFourDivisors(nums) for nums, _ in EXAMPLES])

[32, 64, 0]


**Độ phức tạp Cách 2:** `O(n·√M)` ≈ `10^4 · 316 ≈ 3·10^6` bước — chạy được (khoảng 1 giây trong Python). Bộ nhớ `O(1)`.

Mẹo nhỏ: có thể **cache** kết quả theo `x` (dict) vì `nums` có thể lặp số — Ví dụ 2 có hai số 21.

### 7.2 Cách 3 — Sàng SPF + nhận dạng `p³` hoặc `p·q`

**Từng đoạn:**
1. Sàng `spf` tới `max(nums)` (một lần).
2. Với mỗi `x > 1`: `p = spf[x]`, `q = x // p`.
3. Kiểm tra hai dạng như mục 6.

In [11]:
SPF = build_spf(MAXV)

def four_div_sum_spf(x, spf=SPF):
    if x < 6:                               # số nhỏ nhất có 4 ước là 6 = 2·3 (và 8 = 2³)
        return 0
    p = spf[x]
    q = x // p
    if p < q and spf[q] == q:               # x = p·q, q nguyên tố lớn hơn p (p < q loại cả q = 1)
        return 1 + p + q + x
    if q == p * p:                          # x = p³
        return 1 + p + q + x
    return 0

def sum_four_spf(nums):
    return sum(four_div_sum_spf(x) for x in nums)

for nums, want in EXAMPLES:
    print(sum_four_spf(nums), "(mong đợi", want, ")")
print("8 ->", four_div_sum_spf(8), "| 12 ->", four_div_sum_spf(12), "| 49 ->", four_div_sum_spf(49))

32 (mong đợi 32 )
64 (mong đợi 64 )
0 (mong đợi 0 )
8 -> 15 | 12 -> 0 | 49 -> 0


Giải thích nhánh `p³`: `p = spf[x]` là ước nguyên tố nhỏ nhất. Nếu `x = p³` thì `q = x // p = p²`. Ngược lại nếu `q = p²` thì `x = p³`. Ở nhánh này tổng ước là `1 + p + p² + p³ = 1 + p + q + x` — trùng công thức với nhánh `p·q`!

Tại sao nhánh `p·q` chỉ cần `q` nguyên tố? Vì `p` là ước nguyên tố **nhỏ nhất** nên với `x` hợp số thì `p ≤ q`; `q` nguyên tố và `q > p` ⇒ `x` là tích hai nguyên tố phân biệt. Nếu `x` nguyên tố thì `p = x`, `q = 1 < p` ⇒ bị loại đúng.

Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def sumFourDivisors(self, nums: list[int]) -> int:
        M = max(nums)
        spf = list(range(M + 1))
        for p in range(2, math.isqrt(M) + 1):
            if spf[p] == p:
                for m in range(p * p, M + 1, p):
                    if spf[m] == m:
                        spf[m] = p
        total = 0
        for x in nums:
            if x < 6:
                continue
            p = spf[x]
            q = x // p
            if (p < q and spf[q] == q) or q == p * p:
                total += 1 + p + q + x
        return total

print([Solution().sumFourDivisors(nums) for nums, _ in EXAMPLES])

[32, 64, 0]


**Độ phức tạp Cách 3:** sàng `O(M log log M)` với `M = max(nums)` + `O(1)` mỗi số ⇒ `O(M log log M + n)`; bộ nhớ `O(M)`.

### 7.3 Cách 4 — Sàng điều hòa: tính sẵn số ước và tổng ước cho mọi số

Tính `cnt[x]` và `sm[x]` cho mọi `x ≤ M` bằng sàng điều hòa; đáp án = `Σ sm[x]` với `cnt[x] == 4`. Đơn giản, không cần lý thuyết số, nhưng tốn `O(M log M)` ≈ `1,2·10^6` phép cho `M = 10^5`.

In [13]:
def build_div_tables(M):
    cnt = [0] * (M + 1)
    sm = [0] * (M + 1)
    for d in range(1, M + 1):
        for m in range(d, M + 1, d):
            cnt[m] += 1
            sm[m] += d
    return cnt, sm

def sum_four_harmonic(nums):
    cnt, sm = build_div_tables(max(nums))
    return sum(sm[x] for x in nums if cnt[x] == 4)

for nums, want in EXAMPLES:
    print(sum_four_harmonic(nums), "(mong đợi", want, ")")

32 (mong đợi 32 )
64 (mong đợi 64 )
0 (mong đợi 0 )


Code nộp LeetCode (Cách 4):

In [14]:
class Solution:
    def sumFourDivisors(self, nums: list[int]) -> int:
        M = max(nums)
        cnt = [0] * (M + 1)
        sm = [0] * (M + 1)
        for d in range(1, M + 1):
            for m in range(d, M + 1, d):
                cnt[m] += 1
                sm[m] += d
        return sum(sm[x] for x in nums if cnt[x] == 4)

print([Solution().sumFourDivisors(nums) for nums, _ in EXAMPLES])

[32, 64, 0]


**Độ phức tạp Cách 4:** `O(M log M + n)` thời gian, `O(M)` bộ nhớ. Hợp khi có **rất nhiều** truy vấn trên cùng miền giá trị.

## 8. TRACE từng bước

**Trace Cách 2:** in từng `d` chia hết, cặp ước, `cnt`, `s`, và điểm dừng sớm, cho các số `21, 4, 7, 12`.

In [15]:
def trace_sqrt(x):
    cnt = s = 0
    d = 1
    print(f"x = {x}, √x ≈ {math.isqrt(x)}")
    while d * d <= x:
        if x % d == 0:
            e = x // d
            if d == e:
                cnt += 1; s += d
                print(f"   d={d}: d*d == x -> thêm 1 ước ({d}); cnt={cnt}, s={s}")
            else:
                cnt += 2; s += d + e
                print(f"   d={d}: cặp ({d}, {e}); cnt={cnt}, s={s}")
            if cnt > 4:
                print("   cnt > 4 -> DỪNG SỚM, đóng góp 0")
                return 0
        d += 1
    res = s if cnt == 4 else 0
    print(f"   kết thúc: cnt={cnt} -> đóng góp {res}")
    return res

total = sum(trace_sqrt(x) for x in [21, 4, 7, 12])
print("tổng =", total)

x = 21, √x ≈ 4
   d=1: cặp (1, 21); cnt=2, s=22
   d=3: cặp (3, 7); cnt=4, s=32
   kết thúc: cnt=4 -> đóng góp 32
x = 4, √x ≈ 2
   d=1: cặp (1, 4); cnt=2, s=5
   d=2: d*d == x -> thêm 1 ước (2); cnt=3, s=7
   kết thúc: cnt=3 -> đóng góp 0
x = 7, √x ≈ 2
   d=1: cặp (1, 7); cnt=2, s=8
   kết thúc: cnt=2 -> đóng góp 0
x = 12, √x ≈ 3
   d=1: cặp (1, 12); cnt=2, s=13
   d=2: cặp (2, 6); cnt=4, s=21
   d=3: cặp (3, 4); cnt=6, s=28
   cnt > 4 -> DỪNG SỚM, đóng góp 0
tổng = 32


**Trace Cách 3:** bảng nhận dạng dạng số bằng SPF:

| x | p = spf[x] | q = x // p | q nguyên tố & q ≠ p? | q == p²? | đóng góp |
|---|---|---|---|---|---|
| 21 | 3 | 7 | ✓ | – | 32 |
| 8 | 2 | 4 | ✗ | ✓ | 15 |
| 12 | 2 | 6 | ✗ | ✗ | 0 |
| 49 | 7 | 7 | ✗ (q = p) | ✗ | 0 |
| 30 | 2 | 15 | ✗ | ✗ | 0 |

In [16]:
def trace_spf(xs):
    for x in xs:
        if x < 6:
            print(f"x={x:3d}: x < 6 -> 0"); continue
        p = SPF[x]; q = x // p
        pq = p < q and SPF[q] == q
        cube = q == p * p
        res = 1 + p + q + x if (pq or cube) else 0
        print(f"x={x:3d}: p={p}, q={q:3d} | p·q? {pq!s:5} | p³? {cube!s:5} | đóng góp {res}")

trace_spf([21, 8, 12, 49, 30, 4, 6, 27, 125, 1])

x= 21: p=3, q=  7 | p·q? True  | p³? False | đóng góp 32
x=  8: p=2, q=  4 | p·q? False | p³? True  | đóng góp 15
x= 12: p=2, q=  6 | p·q? False | p³? False | đóng góp 0
x= 49: p=7, q=  7 | p·q? False | p³? False | đóng góp 0
x= 30: p=2, q= 15 | p·q? False | p³? False | đóng góp 0
x=  4: x < 6 -> 0
x=  6: p=2, q=  3 | p·q? True  | p³? False | đóng góp 12
x= 27: p=3, q=  9 | p·q? False | p³? True  | đóng góp 40
x=125: p=5, q= 25 | p·q? False | p³? True  | đóng góp 156
x=  1: x < 6 -> 0


## 9. Code nộp LeetCode

Bản khuyên dùng: Cách 3 (sàng SPF + nhận dạng dạng số), nhanh và gọn.

In [17]:
import math

class Solution:
    def sumFourDivisors(self, nums: list[int]) -> int:
        M = max(nums)
        spf = list(range(M + 1))                       # sàng ước nguyên tố nhỏ nhất
        for p in range(2, math.isqrt(M) + 1):
            if spf[p] == p:
                for m in range(p * p, M + 1, p):
                    if spf[m] == m:
                        spf[m] = p
        total = 0
        for x in nums:
            if x < 6:                                  # 1..5 không có 4 ước
                continue
            p = spf[x]
            q = x // p
            if (p < q and spf[q] == q) or q == p * p:  # x = p·q hoặc x = p³
                total += 1 + p + q + x
        return total

for nums, want in EXAMPLES:
    got = Solution().sumFourDivisors(nums)
    print(got, "OK" if got == want else "SAI")

32 OK
64 OK
0 OK


## 10. Vì sao thuật toán đúng

**Cách 2:** mọi ước `e > √x` đi kèm đúng một ước `d = x // e < √x`, nên duyệt `d ≤ √x` và thêm `x // d` liệt kê **đủ và không trùng** mọi ước (trường hợp `d = x // d` chỉ thêm một lần). Dừng sớm khi `cnt > 4` không làm sai vì `cnt` chỉ tăng.

**Cách 3:** theo công thức `τ(x) = Π(ai + 1)`, `τ(x) = 4` ⇔ (một số mũ bằng 3, không có thừa số khác) hoặc (hai số mũ bằng 1) ⇔ `x = p³` hoặc `x = p·q`, `p ≠ q`.
- Nếu `x = p·q` với `p < q` nguyên tố thì `spf[x] = p`, `x // p = q` nguyên tố, `q ≠ p` ⇒ nhánh 1 bắt được.
- Nếu `x = p³` thì `spf[x] = p`, `x // p = p²` ⇒ nhánh 2 bắt được.
- Ngược lại: nếu nhánh 1 đúng thì `x = p·q` hai nguyên tố phân biệt; nếu nhánh 2 đúng thì `x = p·p² = p³`. Vậy không có dương tính giả.

Tổng ước: `1 + p + q + pq` cho dạng `p·q`, `1 + p + p² + p³` cho dạng `p³` — cả hai đều là `1 + p + q + x` với `q = x // p`.

**Cách 4:** `d` được cộng vào `cnt[m]`, `sm[m]` đúng khi `d | m` ⇒ `cnt`, `sm` là số ước và tổng ước chính xác.

## 11. Phân tích Time / Space

Gọi `n = len(nums) ≤ 10^4`, `M = max(nums) ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `d ≤ x` | `O(n·M)` ≈ `10^9` | `O(1)` (hoặc `O(τ)` cho list ước) |
| 2. Tới `√x` + dừng sớm | `O(n·√M)` ≈ `3·10^6` | `O(1)` |
| 3. Sàng SPF + nhận dạng | `O(M log log M + n)` | `O(M)` |
| 4. Sàng điều hòa | `O(M log M + n)` | `O(M)` |

- Worst-case Cách 2 là khi các số là **nguyên tố lớn** hoặc `p·q` với `p` lớn (không dừng sớm, phải đi hết `√x`). Số có nhiều ước nhỏ (như `2^k·3`) dừng rất sớm.
- Cách 3/4 tốn bộ nhớ `O(M)` nhưng mỗi số chỉ `O(1)`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "brute": sum_four_brute,
    "sqrt": sum_four_sqrt,
    "spf": sum_four_spf,
    "harmonic": sum_four_harmonic,
    "submit": lambda nums: Solution().sumFourDivisors(nums),
}
edge = [
    ([1], 0),
    ([6], 12),                 # 6 = 2·3: 1+2+3+6
    ([8], 15),                 # 8 = 2³
    ([27], 40),                # 27 = 3³: 1+3+9+27
    ([4, 9, 25, 49], 0),       # p²: 3 ước
    ([16], 0),                 # 2^4: 5 ước
    ([12, 30, 60], 0),         # nhiều ước
    ([99991], 0),              # số nguyên tố lớn
    ([99998], 1 + 2 + 49999 + 99998),   # 2 · 49999 (49999 nguyên tố)
    ([6, 6, 8], 39),           # lặp số
]
for name, fn in METHODS.items():
    for nums, want in EXAMPLES + edge:
        got = fn(nums)
        assert got == want, (name, nums, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 13 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định) và vét cạn `1..5000`

In [19]:
rng = random.Random(1390)
for _ in range(300):
    nums = [rng.randint(1, 3000) for _ in range(rng.randint(1, 20))]
    res = {name: fn(nums) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, (nums, res)
print("300 mảng ngẫu nhiên (giá trị ≤ 3000): mọi cách khớp nhau")

cnt_t, sm_t = build_div_tables(5000)
for x in range(1, 5001):
    want = sm_t[x] if cnt_t[x] == 4 else 0
    assert four_div_sum(x) == want and four_div_sum_spf(x) == want, x
print("Vét cạn x = 1..5000: Cách 2 và Cách 3 khớp sàng điều hòa")

300 mảng ngẫu nhiên (giá trị ≤ 3000): mọi cách khớp nhau
Vét cạn x = 1..5000: Cách 2 và Cách 3 khớp sàng điều hòa


### 12.3 Test lớn theo ràng buộc: `n = 10^4`, giá trị tới `10^5`

In [20]:
rng = random.Random(7)
big = [rng.randint(1, MAXV) for _ in range(10_000)]
primes_big = [x for x in range(MAXV, 1, -1) if SPF[x] == x][:10_000]   # worst-case cho Cách 2
for name in ["sqrt", "spf", "harmonic", "submit"]:
    t0 = time.perf_counter()
    a = METHODS[name](big)
    b = METHODS[name](primes_big)
    print(f"{name:9s} -> ngẫu nhiên {a}, nguyên tố lớn {b} | {time.perf_counter() - t0:.3f}s")

sqrt      -> ngẫu nhiên 137338494, nguyên tố lớn 0 | 0.145s
spf       -> ngẫu nhiên 137338494, nguyên tố lớn 0 | 0.004s


harmonic  -> ngẫu nhiên 137338494, nguyên tố lớn 0 | 0.222s
submit    -> ngẫu nhiên 137338494, nguyên tố lớn 0 | 0.020s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm hai lần ước `√x` của số chính phương

In [21]:
def wrong_square(x):
    cnt = s = 0
    d = 1
    while d * d <= x:
        if x % d == 0:
            cnt += 2                      # SAI: khi d == x // d chỉ là MỘT ước
            s += d + x // d
        d += 1
    return s if cnt == 4 else 0

for x in [4, 9, 25]:
    print(f"x={x}: sai = {wrong_square(x)} | đúng = {four_div_sum(x)}")

x=4: sai = 9 | đúng = 0
x=9: sai = 16 | đúng = 0
x=25: sai = 36 | đúng = 0


**Vì sao sai:** `4` có cặp `(1,4)` và `(2,2)`; đếm `(2,2)` là 2 ước ⇒ `cnt = 4` ⇒ nhận nhầm 4 là số có 4 ước, cộng `1+4+2+2 = 9`. Thực tế 4 chỉ có 3 ước. Mọi `p²` đều dính lỗi này.

**Code đúng:** `if d == x // d: cnt += 1 else: cnt += 2`.

### Lỗi 2 — Chỉ nhận dạng dạng `p·q`, quên dạng `p³`

In [22]:
def wrong_no_cube(nums):
    total = 0
    for x in nums:
        if x < 6: continue
        p = SPF[x]; q = x // p
        if p < q and SPF[q] == q:         # SAI: thiếu nhánh x = p³
            total += 1 + p + q + x
    return total

for nums in ([8, 21], [27], [125, 6]):
    print(nums, "-> sai:", wrong_no_cube(nums), "| đúng:", sum_four_spf(nums))

[8, 21] -> sai: 32 | đúng: 47
[27] -> sai: 0 | đúng: 40
[125, 6] -> sai: 12 | đúng: 168


**Vì sao sai:** `8 = 2³` có 4 ước `1,2,4,8` nhưng không phải tích hai nguyên tố phân biệt ⇒ bị bỏ sót.

**Code đúng:** thêm `or q == p * p`.

### Lỗi 3 — Cho phép `p == q` trong dạng `p·q`

In [23]:
def wrong_allow_equal(nums):
    total = 0
    for x in nums:
        if x < 4: continue
        p = SPF[x]; q = x // p
        if (p <= q and SPF[q] == q) or q == p * p:   # SAI: cho phép q == p
            total += 1 + p + q + x
    return total

for nums in ([9], [49, 21], [4]):
    print(nums, "-> sai:", wrong_allow_equal(nums), "| đúng:", sum_four_spf(nums))

[9] -> sai: 16 | đúng: 0
[49, 21] -> sai: 96 | đúng: 32
[4] -> sai: 9 | đúng: 0


**Vì sao sai:** `9 = 3·3` có `q = 3` nguyên tố nhưng `q == p` ⇒ 9 có 3 ước `1,3,9`, không phải 4. Code sai cộng `1+3+3+9 = 16`.

**Code đúng:** `p < q and spf[q] == q` (vì `p` là ước nguyên tố nhỏ nhất nên `p ≤ q`; loại `q == p`).

### Lỗi 4 — Dừng sớm với điều kiện `cnt >= 4`

In [24]:
def wrong_early_stop(x):
    cnt = s = 0
    d = 1
    while d * d <= x:
        if x % d == 0:
            e = x // d
            cnt += 1 if d == e else 2
            s += d if d == e else d + e
            if cnt >= 4:                  # SAI: dừng khi mới ĐỦ 4, chưa biết còn ước nữa không
                break
        d += 1
    return s if cnt == 4 else 0

for x in [12, 30, 21]:
    print(f"x={x}: sai = {wrong_early_stop(x)} | đúng = {four_div_sum(x)}")

x=12: sai = 21 | đúng = 0
x=30: sai = 48 | đúng = 0
x=21: sai = 32 | đúng = 32


**Vì sao sai:** với `12`, sau `d = 1, 2` đã có 4 ước `1,12,2,6` ⇒ dừng và kết luận "đúng 4 ước", bỏ sót `3, 4`. Chỉ được dừng khi chắc chắn **vượt** 4.

**Code đúng:** `if cnt > 4: return 0`.

### Lỗi 5 — Nhận dạng `p·q` bằng `q != p and spf[q] == q` (số nguyên tố lọt lưới)

In [25]:
def wrong_prime_slip(nums):
    total = 0
    for x in nums:
        if x < 6: continue
        p = SPF[x]; q = x // p
        if (q != p and SPF[q] == q) or q == p * p:   # SAI: x nguyên tố -> p = x, q = 1, spf[1] == 1 !
            total += 1 + p + q + x
    return total

for nums in ([7], [21, 4, 7], [99991]):
    print(nums, "-> sai:", wrong_prime_slip(nums), "| đúng:", sum_four_spf(nums))

[7] -> sai: 16 | đúng: 0
[21, 4, 7] -> sai: 48 | đúng: 32
[99991] -> sai: 199984 | đúng: 0


**Vì sao sai:** với `x` nguyên tố, `spf[x] = x` nên `q = x // x = 1`; mảng `spf` có `spf[1] = 1` nên điều kiện "`q` nguyên tố" **đúng một cách giả tạo**, và `q != p` cũng đúng ⇒ số nguyên tố bị cộng `1 + x + 1 + x`. Ví dụ 1 trả 48 thay vì 32 (do số 7). Lỗi này lọt qua nếu chỉ test bằng các số hợp số.

**Code đúng:** `p < q and spf[q] == q` (hoặc thêm `q > 1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Liệt kê/đếm ước"** ⇒ duyệt tới `√x`, nhớ trường hợp `d·d == x`.
- **"Đúng k ước"** với `k` nhỏ ⇒ nghĩ tới công thức `τ = Π(ai+1)`: `k = 2` ⇒ nguyên tố; `k = 3` ⇒ `p²`; `k = 4` ⇒ `p³` hoặc `p·q`.
- **Nhiều số, giá trị ≤ 10^6** ⇒ sàng (Eratosthenes / SPF / điều hòa) một lần, mỗi số `O(1)` hoặc `O(log x)`.
- Có điều kiện "chỉ cần biết có đúng k hay không" ⇒ **dừng sớm** khi vượt.
- Cache theo giá trị khi `nums` có nhiều số lặp.

## 15. Biến thể của bài

1. **Đúng 3 ước** (LeetCode 1952 — Three Divisors): `x = p²` ⇔ `√x` nguyên và nguyên tố.
2. **Đúng `k` ước bất kỳ**: sàng điều hòa cho `cnt[x]`, `sm[x]` (bài tập 5).
3. **Đếm số `≤ N` có đúng 4 ước** với `N` lớn: đếm `p³ ≤ N` + số cặp `p < q` với `p·q ≤ N` (dùng danh sách nguyên tố và hai con trỏ).
4. **LeetCode 204 — Count Primes:** sàng Eratosthenes cơ bản.
5. **LeetCode 2521 — Distinct Prime Factors of Product of Array:** phân tích thừa số bằng SPF.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `d` | đếm ước ngây thơ | `O(n·M)` | `O(1)` | kiểm chứng, số nhỏ |
| 2. Tới `√x` + dừng sớm | cặp ước, bỏ khi > 4 | `O(n·√M)` | `O(1)` | dễ viết, không cần lý thuyết số |
| 3. SPF + nhận dạng | `p³` hoặc `p·q` | `O(M log log M + n)` | `O(M)` | **khuyên dùng**, nhanh nhất |
| 4. Sàng điều hòa | tính sẵn `τ`, `σ` mọi số | `O(M log M + n)` | `O(M)` | nhiều truy vấn, tổng quát cho mọi `k` |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `sumFourDivisors([8, 10, 12])` và `sumFourDivisors([6, 6, 6])`?

**Bài 2 (tính tay):** liệt kê ước của `21`, `49`, `30`, `125` và cho biết số nào có đúng 4 ước.

**Bài 3 (điền TODO):** hoàn thiện sàng SPF.

In [26]:
def build_spf_todo(M):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    spf = list(range(M + 1))
    for p in range(2, math.isqrt(M) + 1):
        # TODO: nếu p nguyên tố (spf[p] == p): với mọi bội m của p từ p*p,
        #       nếu spf[m] chưa bị gán (spf[m] == m) thì gán spf[m] = p
        pass
    return spf

**Bài 4 (tự viết lại):** viết lại Cách 2 có **cache** theo giá trị (dict), và đo thời gian trên mảng `10^4` phần tử chỉ gồm 100 giá trị khác nhau.

**Bài 5 (mini challenge):** viết `sum_k_divisors(nums, k)`: tổng các ước của những số trong `nums` có **đúng `k` ước** (`1 ≤ k ≤ 128`).

In [27]:
def sum_k_divisors(nums, k):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | 8 = 2³, 10 = 2·5, 12 có 6 ước | 15 + 18 | 33 và 36 |
| 2 | phân tích thừa số | 21 = 3·7, 49 = 7², 30 = 2·3·5, 125 = 5³ | 21 và 125 có 4 ước |
| 3 | vòng ngoài chỉ tới `√M` | `range(p*p, M+1, p)` | `if spf[m] == m: spf[m] = p` |
| 4 | `cache = {}` | `if x not in cache: cache[x] = four_div_sum(x)` | cộng `cache[x]` |
| 5 | sàng điều hòa tới `max(nums)` | `cnt[x] == k` | `sum(sm[x] for x in nums if cnt[x] == k)` |

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ans1 = [sum_four_spf([8, 10, 12]), sum_four_spf([6, 6, 6])]
print("Bài 1:", ans1)
assert ans1 == [33, 36]

# Bài 2
ans2 = {x: [d for d in range(1, x + 1) if x % d == 0] for x in (21, 49, 30, 125)}
four = [x for x, ds in ans2.items() if len(ds) == 4]
print("Bài 2:", ans2, "| có đúng 4 ước:", four)
assert four == [21, 125]

Bài 1: [33, 36]
Bài 2: {21: [1, 3, 7, 21], 49: [1, 7, 49], 30: [1, 2, 3, 5, 6, 10, 15, 30], 125: [1, 5, 25, 125]} | có đúng 4 ước: [21, 125]


In [29]:
# Bài 3
def build_spf_sol(M):
    spf = list(range(M + 1))
    for p in range(2, math.isqrt(M) + 1):
        if spf[p] == p:
            for m in range(p * p, M + 1, p):
                if spf[m] == m:
                    spf[m] = p
    return spf

s3 = build_spf_sol(10_000)
assert s3 == SPF[:10_001]
assert all(s3[x] == min(factorize(x)) for x in range(2, 2001))
print("Bài 3: OK")

Bài 3: OK


In [30]:
# Bài 4
def sum_four_cached(nums):
    cache, total = {}, 0
    for x in nums:
        if x not in cache:
            cache[x] = four_div_sum(x)
        total += cache[x]
    return total

rng = random.Random(4)
pool = [rng.randint(1, MAXV) for _ in range(100)]
arr = [rng.choice(pool) for _ in range(10_000)]
t0 = time.perf_counter(); r_a = sum_four_cached(arr); t_a = time.perf_counter() - t0
t0 = time.perf_counter(); r_b = sum_four_sqrt(arr); t_b = time.perf_counter() - t0
print(f"Bài 4: cache {t_a:.3f}s, không cache {t_b:.3f}s")
assert r_a == r_b == sum_four_spf(arr)

Bài 4: cache 0.001s, không cache 0.035s


In [31]:
# Bài 5
def sum_k_divisors_sol(nums, k):
    cnt, sm = build_div_tables(max(nums))
    return sum(sm[x] for x in nums if cnt[x] == k)

assert sum_k_divisors_sol([21, 4, 7], 4) == 32
assert sum_k_divisors_sol([4, 9, 10], 3) == (1 + 2 + 4) + (1 + 3 + 9)
assert sum_k_divisors_sol([2, 3, 4], 2) == 3 + 4
rng = random.Random(5)
for _ in range(100):
    nums = [rng.randint(1, 500) for _ in range(10)]
    k = rng.randint(1, 12)
    want = sum(sum(d for d in range(1, x + 1) if x % d == 0) for x in nums
               if sum(1 for d in range(1, x + 1) if x % d == 0) == k)
    assert sum_k_divisors_sol(nums, k) == want
assert all(sum_k_divisors_sol(n, 4) == sum_four_brute(n) for n, _ in EXAMPLES)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template đếm ước tới √x (dừng sớm):**

```python
cnt = s = 0; d = 1
while d * d <= x:
    if x % d == 0:
        e = x // d
        if d == e: cnt += 1; s += d        # số chính phương: 1 ước
        else:      cnt += 2; s += d + e
        if cnt > 4: break                   # VƯỢT 4 mới dừng
    d += 1
```

**Template nhận dạng bằng SPF:** `p = spf[x]; q = x // p; if (p < q and spf[q] == q) or q == p*p: total += 1 + p + q + x`.

**Checklist trước khi submit:**
- [ ] `d·d == x` chỉ đếm một ước.
- [ ] Dừng sớm khi `cnt > 4`, không phải `>= 4`.
- [ ] Nhận dạng đủ hai dạng `p·q` (`p < q`) **và** `p³`; số nguyên tố (`q = 1`) bị loại.
- [ ] Số `1..5` không có 4 ước (bỏ qua trước khi tra `spf`).
- [ ] Số lặp lại thì cộng lại (Ví dụ 2 → 64).
- [ ] Đúng chữ ký: `sumFourDivisors(self, nums: list[int]) -> int`.